In [ ]:
# Fine-tunes VideoMAE again on the extended Training2 dataset.

# Fine-tuning VideoMAE avec `Training2`

Ce notebook reprend la méthode exacte de l’ancien entraînement :

- les classes restent organisées directement en dossiers ;
- le split historique est reconstruit depuis l’ancien dataset `Training` avec `SEED = 42` et `CLIPS_PER_SOURCE = 5` ;
- les anciens jeux de validation et de test restent inchangés ;
- les 55 nouvelles vidéos détectées dans `Training2` sont ajoutées uniquement au train ;
- le même mapping des 12 classes est conservé ;
- le fine-tuning démarre depuis l’ancien `checkpoint-best.pth`.

Dans Kaggle, ajoute comme inputs :

1. l’ancien dataset contenant `Training` ;
2. le nouveau dataset contenant `Training2` ;
3. le dataset contenant l’ancien `checkpoint-best.pth`.

Active deux GPU T4 avant de lancer le notebook.


In [ ]:
# VIDEOMAE — CELLULE 2
# Création d'un environnement Python 3.10 isolé

from pathlib import Path
import shutil
import subprocess
import sys

ENV_DIR = Path("/kaggle/working/vmae_env")

# Installer uv, utilisé pour créer rapidement l'environnement
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "uv",
    ],
    check=True,
)

UV = shutil.which("uv")

if UV is None:
    raise FileNotFoundError(
        "La commande uv n'a pas été trouvée."
    )

print("uv trouvé :", UV)

# Supprimer uniquement un éventuel ancien environnement VideoMAE
shutil.rmtree(
    ENV_DIR,
    ignore_errors=True,
)

# Installer Python 3.10
subprocess.run(
    [
        UV,
        "python",
        "install",
        "3.10",
    ],
    check=True,
)

# Créer l'environnement isolé
subprocess.run(
    [
        UV,
        "venv",
        "--python",
        "3.10",
        str(ENV_DIR),
    ],
    check=True,
)

PY = str(ENV_DIR / "bin" / "python")

assert Path(PY).exists(), (
    f"Python absent : {PY}"
)

# Vérification
subprocess.run(
    [
        PY,
        "--version",
    ],
    check=True,
)

print("Environnement créé :", ENV_DIR)
print("Python utilisé :", PY)
print("ENVIRONNEMENT VIDEOMAE VALIDÉ")

In [ ]:
# VIDEOMAE — CELLULE 3
# Installation de PyTorch et des dépendances VideoMAEv2

from pathlib import Path
import shutil
import subprocess

ENV_DIR = Path("/kaggle/working/vmae_env")
PY = str(ENV_DIR / "bin" / "python")

assert Path(PY).exists(), (
    f"Python de l'environnement absent : {PY}"
)

UV = shutil.which("uv")

if UV is None:
    raise FileNotFoundError(
        "La commande uv est introuvable."
    )

print("=" * 70)
print("1. INSTALLATION DE PYTORCH + CUDA 11.8")
print("=" * 70)

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--python",
        PY,
        "--index-url",
        "https://download.pytorch.org/whl/cu118",
        "torch==2.1.2",
        "torchvision==0.16.2",
        "torchaudio==2.1.2",
    ],
    check=True,
)

print("\n" + "=" * 70)
print("2. INSTALLATION DES DÉPENDANCES VIDEOMAE")
print("=" * 70)

subprocess.run(
    [
        UV,
        "pip",
        "install",
        "--python",
        PY,

        # NumPy 1.x pour éviter les incompatibilités
        # avec les anciennes bibliothèques VideoMAE
        "numpy==1.26.4",

        # Version recommandée par VideoMAEv2
        "timm==0.4.12",

        # Lecture et traitement vidéo
        "decord==0.6.0",
        "av",
        "opencv-python-headless",
        "Pillow",

        # Calcul scientifique
        "einops",
        "scipy",
        "pandas",

        # Logs et résultats
        "tensorboardX",
        "matplotlib",
        "scikit-learn",

        # Téléchargement du modèle préentraîné
        "huggingface_hub",
    ],
    check=True,
)

print("\n" + "=" * 70)
print("3. VÉRIFICATION DES INSTALLATIONS")
print("=" * 70)

verification_code = r"""
import sys

import torch
import torchvision
import timm
import numpy
import decord
import av
import cv2
import einops
import scipy
import pandas
import tensorboardX
import sklearn
import huggingface_hub

print("Python       :", sys.version.split()[0])
print("PyTorch      :", torch.__version__)
print("Torchvision  :", torchvision.__version__)
print("CUDA PyTorch :", torch.version.cuda)
print("CUDA dispo   :", torch.cuda.is_available())
print("Nombre GPU   :", torch.cuda.device_count())

for index in range(torch.cuda.device_count()):
    print(
        f"GPU {index}       :",
        torch.cuda.get_device_name(index)
    )

print("timm          :", timm.__version__)
print("NumPy         :", numpy.__version__)
print("decord        :", decord.__version__)
print("PyAV          :", av.__version__)
print("OpenCV        :", cv2.__version__)
print("SciPy         :", scipy.__version__)
print("Pandas        :", pandas.__version__)

# Petit calcul GPU réel
if not torch.cuda.is_available():
    raise RuntimeError("CUDA n'est pas disponible dans l'environnement.")

x = torch.randn(
    128,
    128,
    device="cuda",
)

y = x @ x.T

assert y.shape == (128, 128)
assert torch.isfinite(y).all()

print("Test calcul GPU : VALIDÉ")
print("DÉPENDANCES VIDEOMAE VALIDÉES")
"""

subprocess.run(
    [
        PY,
        "-c",
        verification_code,
    ],
    check=True,
)

In [ ]:
# VIDEOMAE — CELLULE 4A
# Télécharger exactement la version utilisée du dépôt

from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path("/kaggle/working/VideoMAEv2")
COMMIT = "29eab1e8a588d1b3ec0cdec7b03a86cca491b74b"

shutil.rmtree(REPO_DIR, ignore_errors=True)

subprocess.run(
    [
        "git",
        "clone",
        "https://github.com/OpenGVLab/VideoMAEv2.git",
        str(REPO_DIR),
    ],
    check=True,
)

subprocess.run(
    ["git", "checkout", COMMIT],
    cwd=str(REPO_DIR),
    check=True,
)

actual_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=str(REPO_DIR),
    text=True,
).strip()

print("Commit téléchargé :", actual_commit)
assert actual_commit == COMMIT
print("DÉPÔT VIDEOMAEV2 VALIDÉ")


In [ ]:
# VIDEOMAE — CELLULE 4B
# Vérification du dépôt et création du modèle

from pathlib import Path
import subprocess

REPO_DIR = Path("/kaggle/working/VideoMAEv2")
PY = "/kaggle/working/vmae_env/bin/python"

assert REPO_DIR.exists(), (
    f"Dépôt absent : {REPO_DIR}"
)

assert Path(PY).exists(), (
    f"Environnement Python absent : {PY}"
)

# ============================================================
# 1. Vérification des vrais chemins du dépôt
# ============================================================

required_files = [
    REPO_DIR / "run_class_finetuning.py",
    REPO_DIR / "dataset" / "build.py",
    REPO_DIR / "dataset" / "datasets.py",
    REPO_DIR / "models" / "__init__.py",
    REPO_DIR / "models" / "modeling_finetune.py",
    REPO_DIR / "utils.py",
]

print("=" * 70)
print("1. FICHIERS ESSENTIELS")
print("=" * 70)

for file_path in required_files:
    exists = file_path.exists()

    print(
        file_path.relative_to(REPO_DIR),
        "->",
        exists,
    )

    assert exists, (
        f"Fichier essentiel absent : {file_path}"
    )

# ============================================================
# 2. Diagnostic du script d'entraînement
# ============================================================

training_script = (
    REPO_DIR / "run_class_finetuning.py"
).read_text(
    encoding="utf-8"
)

dataset_build = (
    REPO_DIR / "dataset" / "build.py"
).read_text(
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("2. DIAGNOSTIC DU SCRIPT")
print("=" * 70)

print(
    "Import obligatoire de DeepSpeed :",
    "import deepspeed" in training_script,
)

print(
    "Option --finetune présente :",
    "--finetune" in training_script,
)

print(
    "Option --resume présente :",
    "--resume" in training_script,
)

print(
    "Option --save_ckpt présente :",
    "--save_ckpt" in training_script,
)

print(
    "Dataset custom déjà supporté :",
    "'custom'" in dataset_build
    or '"custom"' in dataset_build,
)

# ============================================================
# 3. Test de création de VideoMAE
# ============================================================

print("\n" + "=" * 70)
print("3. TEST DE CRÉATION DU MODÈLE")
print("=" * 70)

test_code = r"""
import torch
import timm

# Cet import enregistre les modèles VideoMAE dans timm
import models

model = timm.create_model(
    "vit_base_patch16_224",
    pretrained=False,
    num_classes=12,
    all_frames=16,
    tubelet_size=2,
    drop_path_rate=0.1,
    use_mean_pooling=True,
    init_scale=0.001,
)

number_of_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

classifier = model.get_classifier()

print("Modèle créé :", model.__class__.__name__)
print("Nombre de paramètres :", f"{number_of_parameters:,}")
print(
    "Nombre de classes :",
    getattr(classifier, "out_features", "non détecté"),
)
print("Frames :", 16)
print("Tubelet size :", 2)

assert getattr(classifier, "out_features", None) == 12

print("IMPORT ET CRÉATION VIDEOMAE VALIDÉS")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)

print(result.stdout)

if result.stderr:
    print("Messages système / erreurs :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test de création VideoMAE a échoué."
    )

print("\nDÉPÔT VIDEOMAEV2 VALIDÉ")

In [ ]:
# VIDEOMAE — CELLULE 5
# Adaptation du dépôt à RHAD, sans DeepSpeed

from pathlib import Path
import shutil
import subprocess

REPO_DIR = Path("/kaggle/working/VideoMAEv2")
PY = "/kaggle/working/vmae_env/bin/python"

RUN_FILE = REPO_DIR / "run_class_finetuning.py"
BUILD_FILE = REPO_DIR / "dataset" / "build.py"

assert RUN_FILE.exists(), RUN_FILE
assert BUILD_FILE.exists(), BUILD_FILE
assert Path(PY).exists(), PY

# ============================================================
# 1. SAUVEGARDES DU CODE ORIGINAL
# ============================================================

for file_path in [RUN_FILE, BUILD_FILE]:
    backup_path = file_path.with_suffix(
        file_path.suffix + ".original"
    )

    if not backup_path.exists():
        shutil.copy2(
            file_path,
            backup_path,
        )

        print(
            "Sauvegarde créée :",
            backup_path,
        )
    else:
        print(
            "Sauvegarde déjà présente :",
            backup_path,
        )

# ============================================================
# 2. RENDRE DEEPSPEED OPTIONNEL
# ============================================================

run_text = RUN_FILE.read_text(
    encoding="utf-8"
)

if (
    "try:\n    import deepspeed"
    not in run_text
):
    assert "import deepspeed" in run_text

    run_text = run_text.replace(
        "import deepspeed",
        """try:
    import deepspeed
except ImportError:
    deepspeed = None""",
        1,
    )

# Sécuriser l'option --enable_deepspeed
old_deepspeed_block = """    if known_args.enable_deepspeed:
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""

new_deepspeed_block = """    if known_args.enable_deepspeed:
        if deepspeed is None:
            raise ImportError(
                "DeepSpeed a été demandé mais il n'est pas installé."
            )
        parser = deepspeed.add_config_arguments(parser)
        ds_init = deepspeed.initialize
    else:
        ds_init = None
"""

if old_deepspeed_block in run_text:
    run_text = run_text.replace(
        old_deepspeed_block,
        new_deepspeed_block,
        1,
    )

# ============================================================
# 3. AJOUTER RHAD AUX DATASETS AUTORISÉS
# ============================================================

data_argument_start = run_text.find(
    "parser.add_argument(\n"
    "        '--data_set'"
)

assert data_argument_start != -1, (
    "Argument --data_set introuvable."
)

data_argument_end = run_text.find(
    "type=str",
    data_argument_start,
)

assert data_argument_end != -1

data_argument_block = run_text[
    data_argument_start:data_argument_end
]

if "'RHAD'" not in data_argument_block:
    assert "'MIT'" in data_argument_block

    patched_argument_block = (
        data_argument_block.replace(
            "'MIT'",
            "'MIT', 'RHAD'",
            1,
        )
    )

    run_text = (
        run_text[:data_argument_start]
        + patched_argument_block
        + run_text[data_argument_end:]
    )

RUN_FILE.write_text(
    run_text,
    encoding="utf-8",
)

# ============================================================
# 4. UTILISER TEST.CSV POUR LE VRAI TEST
# ============================================================

build_text = BUILD_FILE.read_text(
    encoding="utf-8"
)

old_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'val.csv')
"""

new_test_block = """    elif test_mode:
        mode = 'test'
        anno_path = os.path.join(args.data_path, 'test.csv')
"""

if old_test_block in build_text:
    build_text = build_text.replace(
        old_test_block,
        new_test_block,
        1,
    )

# ============================================================
# 5. AJOUTER LE DATASET RHAD : 12 CLASSES
# ============================================================

rhad_branch = """
    elif args.data_set == 'RHAD':
        dataset = VideoClsDataset(
            anno_path=anno_path,
            data_root=args.data_root,
            mode=mode,
            clip_len=args.num_frames,
            frame_sample_rate=args.sampling_rate,
            num_segment=1,
            test_num_segment=args.test_num_segment,
            test_num_crop=args.test_num_crop,
            num_crop=1 if not test_mode else 3,
            keep_aspect_ratio=True,
            crop_size=args.input_size,
            short_side_size=args.short_side_size,
            new_height=256,
            new_width=320,
            sparse_sample=False,
            args=args,
        )
        nb_classes = 12
"""

unsupported_block = """    else:
        raise NotImplementedError('Unsupported Dataset')
"""

if "elif args.data_set == 'RHAD':" not in build_text:
    assert unsupported_block in build_text, (
        "Bloc Unsupported Dataset introuvable."
    )

    build_text = build_text.replace(
        unsupported_block,
        rhad_branch + "\n" + unsupported_block,
        1,
    )

BUILD_FILE.write_text(
    build_text,
    encoding="utf-8",
)

# ============================================================
# 6. VÉRIFIER LES PATCHS
# ============================================================

final_run_text = RUN_FILE.read_text(
    encoding="utf-8"
)

final_build_text = BUILD_FILE.read_text(
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("VÉRIFICATION DES MODIFICATIONS")
print("=" * 70)

checks = {
    "DeepSpeed optionnel":
        "except ImportError" in final_run_text,

    "RHAD autorisé dans les arguments":
        "'RHAD'" in final_run_text,

    "RHAD présent dans build.py":
        "elif args.data_set == 'RHAD':"
        in final_build_text,

    "RHAD possède 12 classes":
        "nb_classes = 12"
        in final_build_text,

    "Le test utilise test.csv":
        "anno_path = os.path.join("
        "args.data_path, 'test.csv')"
        in final_build_text,
}

for name, result in checks.items():
    print(f"{name:35s} :", result)
    assert result, name

# ============================================================
# 7. VÉRIFICATION DE LA SYNTAXE PYTHON
# ============================================================

subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(RUN_FILE),
        str(BUILD_FILE),
    ],
    check=True,
)

# Vérifier que les imports fonctionnent sans DeepSpeed
test_code = r"""
import run_class_finetuning
from dataset import build_dataset

print("Import run_class_finetuning : OK")
print("Import build_dataset : OK")
print("PATCH VIDEOMAE RHAD VALIDÉ")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd=str(REPO_DIR),
    text=True,
    capture_output=True,
)

print("\nSortie du test :")
print(result.stdout)

if result.stderr:
    print("Messages système :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test des fichiers modifiés a échoué."
    )

print("\nADAPTATION DU DÉPÔT TERMINÉE")

In [ ]:
# VIDEOMAE — CELLULE 9
# Conserver uniquement checkpoint-latest et checkpoint-best

from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"

UTILS_FILE = Path(
    "/kaggle/working/VideoMAEv2/utils.py"
)

assert Path(PY).exists(), PY
assert UTILS_FILE.exists(), UTILS_FILE

# ============================================================
# 1. SAUVEGARDE DU FICHIER ORIGINAL
# ============================================================

BACKUP_FILE = UTILS_FILE.with_suffix(
    ".py.original"
)

if not BACKUP_FILE.exists():
    shutil.copy2(
        UTILS_FILE,
        BACKUP_FILE,
    )
    print("Sauvegarde créée :", BACKUP_FILE)
else:
    print("Sauvegarde déjà présente :", BACKUP_FILE)

# ============================================================
# 2. MODIFICATION DU NOM DES CHECKPOINTS
# ============================================================

text = UTILS_FILE.read_text(
    encoding="utf-8"
)

old_code = """    if loss_scaler is not None:
        checkpoint_paths = [output_dir / ('checkpoint-%s.pth' % epoch_name)]
"""

new_code = """    if loss_scaler is not None:
        # Le meilleur modèle reste dans checkpoint-best.pth.
        # Tous les checkpoints périodiques écrasent checkpoint-latest.pth
        # afin de ne pas créer des dizaines de fichiers volumineux.
        if epoch_name == "best":
            checkpoint_filename = "checkpoint-best.pth"
        else:
            checkpoint_filename = "checkpoint-latest.pth"

        checkpoint_paths = [
            output_dir / checkpoint_filename
        ]
"""

if old_code in text:
    text = text.replace(
        old_code,
        new_code,
        1,
    )
    UTILS_FILE.write_text(
        text,
        encoding="utf-8",
    )
    print("Patch appliqué.")
elif new_code in text:
    print("Patch déjà appliqué.")
else:
    raise RuntimeError(
        "Le bloc de sauvegarde attendu n'a pas été trouvé."
    )

# ============================================================
# 3. VÉRIFICATIONS
# ============================================================

final_text = UTILS_FILE.read_text(
    encoding="utf-8"
)

checks = {
    "checkpoint-latest présent":
        'checkpoint_filename = "checkpoint-latest.pth"'
        in final_text,

    "checkpoint-best présent":
        'checkpoint_filename = "checkpoint-best.pth"'
        in final_text,

    "ancien fichier numérique supprimé":
        "checkpoint_paths = [output_dir / "
        "('checkpoint-%s.pth' % epoch_name)]"
        not in final_text,
}

print("\n" + "=" * 70)
print("VÉRIFICATION DU PATCH")
print("=" * 70)

for name, result in checks.items():
    print(f"{name:40s} :", result)
    assert result, name

# Vérification de la syntaxe
subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(UTILS_FILE),
    ],
    check=True,
)

# Vérification par import
test_code = r"""
import utils

print("Import utils : OK")
print("GESTION DES CHECKPOINTS VALIDÉE")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    cwd="/kaggle/working/VideoMAEv2",
    text=True,
    capture_output=True,
)

print("\n" + result.stdout)

if result.stderr:
    print("Messages système :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test du patch checkpoints a échoué."
    )

print("\nPRÉPARATION DES CHECKPOINTS TERMINÉE")

In [ ]:
# VIDEOMAE — CELLULE 10A
# Corriger sampler_test

from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"
RUN_FILE = Path(
    "/kaggle/working/VideoMAEv2/run_class_finetuning.py"
)

assert RUN_FILE.exists()
assert Path(PY).exists()

backup = RUN_FILE.with_name(
    "run_class_finetuning.py.before_sampler_patch"
)

if not backup.exists():
    shutil.copy2(RUN_FILE, backup)

text = RUN_FILE.read_text(encoding="utf-8")

old_block = """    else:
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)
"""

new_block = """    else:
        sampler_val = torch.utils.data.SequentialSampler(dataset_val)
        sampler_test = torch.utils.data.DistributedSampler(
            dataset_test,
            num_replicas=num_tasks,
            rank=global_rank,
            shuffle=False,
        )
"""

start = text.index("    if args.dist_eval:")
end = text.index(
    "    if global_rank == 0 and args.log_dir is not None:",
    start,
)
section = text[start:end]
else_section = section.split("    else:", 1)[-1]

if "sampler_test =" not in else_section:
    assert old_block in section
    section = section.replace(old_block, new_block, 1)
    text = text[:start] + section + text[end:]
    RUN_FILE.write_text(text, encoding="utf-8")
    print("Patch sampler_test appliqué.")
else:
    print("Patch sampler_test déjà présent.")

final_text = RUN_FILE.read_text(encoding="utf-8")
start = final_text.index("    if args.dist_eval:")
end = final_text.index(
    "    if global_rank == 0 and args.log_dir is not None:",
    start,
)
final_section = final_text[start:end]

assert "sampler_test =" in final_section.split("    else:", 1)[-1]

subprocess.run(
    [PY, "-m", "py_compile", str(RUN_FILE)],
    check=True,
)

print("SAMPLER_TEST VALIDÉ")


In [ ]:
# VIDEOMAE — CELLULE 10B
# Remplacer le bfloat16 incompatible avec les T4 par du float16

from pathlib import Path
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"

ENGINE_FILE = Path(
    "/kaggle/working/VideoMAEv2/"
    "engine_for_finetuning.py"
)

assert Path(PY).exists(), PY
assert ENGINE_FILE.exists(), ENGINE_FILE

# ============================================================
# 1. SAUVEGARDE
# ============================================================

BACKUP_FILE = ENGINE_FILE.with_name(
    "engine_for_finetuning.py.before_fp16_patch"
)

if not BACKUP_FILE.exists():
    shutil.copy2(
        ENGINE_FILE,
        BACKUP_FILE,
    )
    print("Sauvegarde créée :", BACKUP_FILE)
else:
    print("Sauvegarde déjà présente :", BACKUP_FILE)

# ============================================================
# 2. INSPECTION AVANT MODIFICATION
# ============================================================

text = ENGINE_FILE.read_text(
    encoding="utf-8"
)

bf16_count_before = text.count(
    "torch.bfloat16"
)

print(
    "Occurrences torch.bfloat16 avant patch :",
    bf16_count_before,
)

if bf16_count_before == 0:
    print(
        "Aucune occurrence BF16 : "
        "le patch semble déjà appliqué."
    )
else:
    # Le fichier concerne uniquement le fine-tuning.
    # Toutes ses opérations AMP doivent utiliser FP16 sur T4.
    text = text.replace(
        "torch.bfloat16",
        "torch.float16",
    )

    ENGINE_FILE.write_text(
        text,
        encoding="utf-8",
    )

    print(
        "Toutes les occurrences BF16 ont été "
        "remplacées par FP16."
    )

# ============================================================
# 3. VÉRIFICATION
# ============================================================

final_text = ENGINE_FILE.read_text(
    encoding="utf-8"
)

bf16_count_after = final_text.count(
    "torch.bfloat16"
)

fp16_count = final_text.count(
    "torch.float16"
)

print("\n" + "=" * 70)
print("VÉRIFICATION DU PATCH")
print("=" * 70)

print(
    "Occurrences torch.bfloat16 après patch :",
    bf16_count_after,
)

print(
    "Occurrences torch.float16 :",
    fp16_count,
)

assert bf16_count_after == 0, (
    "Il reste encore une opération en bfloat16."
)

assert fp16_count >= 1, (
    "Aucune opération float16 n'a été trouvée."
)

# Vérifier la syntaxe Python
subprocess.run(
    [
        PY,
        "-m",
        "py_compile",
        str(ENGINE_FILE),
    ],
    check=True,
)

# ============================================================
# 4. TEST AMP RÉEL SUR LE T4
# ============================================================

test_code = r"""
import torch

assert torch.cuda.is_available()

x = torch.randn(
    256,
    256,
    device="cuda",
)

with torch.cuda.amp.autocast(
    dtype=torch.float16
):
    y = x @ x.T

print("GPU :", torch.cuda.get_device_name(0))
print("Type de sortie :", y.dtype)
print("Forme :", tuple(y.shape))
print("Valeurs finies :", torch.isfinite(y).all().item())

assert y.dtype == torch.float16
assert torch.isfinite(y).all()

print("AMP FLOAT16 VALIDÉ SUR TESLA T4")
"""

result = subprocess.run(
    [
        PY,
        "-c",
        test_code,
    ],
    text=True,
    capture_output=True,
)

print("\n" + result.stdout)

if result.stderr:
    print("Messages système :")
    print(result.stderr)

print("Code de retour :", result.returncode)

if result.returncode != 0:
    raise RuntimeError(
        "Le test FP16 sur le GPU a échoué."
    )

print("\nPATCH FP16 VIDEOMAE VALIDÉ")

In [ ]:
# ============================================================
# DÉTECTER TRAINING2 ET L'ANCIEN CHECKPOINT VIDEOMAE
# ============================================================

from pathlib import Path
import torch

KAGGLE_INPUT = Path(
    "/kaggle/input/datasets/nisrineamalkhnifass"
)

# Training2 contient directement un dossier "Training"
training_candidates = [
    path
    for path in KAGGLE_INPUT.rglob("Training")
    if (
        path.is_dir()
        and path.parent.name.lower() == "training2"
    )
]

print("Candidats Training2/Training :")
for path in training_candidates:
    print(" -", path)

assert len(training_candidates) == 1, (
    "Un seul dossier Training2/Training est attendu, "
    f"mais {len(training_candidates)} ont été trouvés."
)

NEW_DATASET = training_candidates[0]

checkpoint_candidates = [
    path
    for path in KAGGLE_INPUT.rglob("checkpoint-best.pth")
    if path.is_file()
]

print("\nCandidats checkpoint-best.pth :")
for path in checkpoint_candidates:
    print(" -", path)

preferred = [
    path
    for path in checkpoint_candidates
    if "videomae-final-rhad-results" in str(path)
]

assert len(preferred) == 1, (
    "Le checkpoint de videomae-final-rhad-results "
    "n'a pas été trouvé de manière unique."
)

OLD_BEST_CHECKPOINT = preferred[0]

print("\n" + "=" * 70)
print("CONFIGURATION RETENUE")
print("=" * 70)
print("Dataset Training2 :", NEW_DATASET)
print("Checkpoint        :", OLD_BEST_CHECKPOINT)

checkpoint = torch.load(
    OLD_BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

weights = checkpoint.get("model", {})

assert "head.weight" in weights
assert tuple(weights["head.weight"].shape) == (12, 768)

print("Epoch du checkpoint :", checkpoint.get("epoch"))
print("Tête du modèle      :", tuple(weights["head.weight"].shape))

del checkpoint


In [ ]:
# ============================================================
# RECONSTRUIRE L'ANCIEN SPLIT ET AJOUTER LES NOUVELLES VIDÉOS
# UNIQUEMENT AU TRAIN
# ============================================================

from pathlib import Path
from collections import defaultdict, Counter
import csv
import json
import os
import random
import re
import shutil

NEW_ROOT = NEW_DATASET

VIDEO_ROOT = Path(
    "/kaggle/working/rhad_videomae_training2_videos"
)

ANNOTATIONS_DIR = Path(
    "/kaggle/working/rhad_videomae_training2_annotations"
)

SEED = 42
CLIPS_PER_SOURCE = 5

VIDEO_EXTENSIONS = {
    ".mp4",
    ".avi",
    ".mkv",
    ".mov",
    ".webm",
}

EXPECTED_MAPPING = {
    "Jumping": 0,
    "child_running": 1,
    "climbing": 2,
    "collesion": 3,
    "electrical_appliances": 4,
    "falling": 5,
    "head_stuck": 6,
    "normal": 7,
    "objects_falling": 8,
    "pulling_curtains": 9,
    "scribbling": 10,
    "sharp_objects": 11,
}

shutil.rmtree(VIDEO_ROOT, ignore_errors=True)
shutil.rmtree(ANNOTATIONS_DIR, ignore_errors=True)

VIDEO_ROOT.mkdir(parents=True, exist_ok=True)
ANNOTATIONS_DIR.mkdir(parents=True, exist_ok=True)

classes = sorted(
    folder.name
    for folder in NEW_ROOT.iterdir()
    if folder.is_dir()
)

class_to_idx = {
    class_name: index
    for index, class_name in enumerate(classes)
}

assert class_to_idx == EXPECTED_MAPPING, (
    f"Mapping incorrect : {class_to_idx}"
)

def list_videos(root: Path):
    return sorted(
        path
        for path in root.rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in VIDEO_EXTENSIONS
        )
    )


new_videos = list_videos(NEW_ROOT)

print("Training2 :", len(new_videos), "vidéos")

assert len(new_videos) == 3303, (
    "Training2 doit contenir 3303 vidéos "
    "(3248 anciennes + 55 nouvelles)."
)

# Les chemins relatifs permettent de repérer les fichiers ajoutés.
added_videos = []

for path in new_videos:
    if path.parent.name != "objects_falling":
        continue

    match = re.fullmatch(
        r"1 \((\d+)\)\.mp4",
        path.name,
    )

    if match:
        number = int(match.group(1))

        if 61 <= number <= 115:
            added_videos.append(path)

added_videos = sorted(
    added_videos,
    key=lambda path: int(
        re.fullmatch(
            r"1 \((\d+)\)\.mp4",
            path.name,
        ).group(1)
    ),
)

print("\nNouvelles vidéos détectées :", len(added_videos))

for path in added_videos[:10]:
    print(" -", path.relative_to(NEW_ROOT))

assert len(added_videos) == 55, (
    f"55 nouvelles vidéos attendues, "
    f"mais {len(added_videos)} ont été détectées."
)

assert all(
    path.relative_to(NEW_ROOT).parts[0] == "objects_falling"
    for path in added_videos
), (
    "Au moins une nouvelle vidéo se trouve dans une autre classe."
)


def extract_group_id(video_path: Path) -> str:
    stem = video_path.stem

    relative_parent = video_path.parent.relative_to(
        NEW_ROOT
    )

    match = re.match(
        r"^(.*?)\s*\((\d+)\)$",
        stem,
    )

    if match:
        prefix = match.group(1).strip()
        clip_number = int(match.group(2))

        block_number = (
            clip_number - 1
        ) // CLIPS_PER_SOURCE

        return (
            f"{relative_parent}/"
            f"{prefix}/block_{block_number:04d}"
        )

    return str(
        video_path.relative_to(NEW_ROOT)
    )


groups_by_class = defaultdict(
    lambda: defaultdict(list)
)

added_video_set = set(added_videos)

for class_name in classes:
    label = class_to_idx[class_name]

    for video_path in sorted(
        path
        for path in (NEW_ROOT / class_name).rglob("*")
        if (
            path.is_file()
            and path.suffix.lower() in VIDEO_EXTENSIONS
            and path not in added_video_set
        )
    ):
        group_id = extract_group_id(video_path)

        groups_by_class[class_name][group_id].append(
            (video_path, label)
        )


rng = random.Random(SEED)

train_samples = []
val_samples = []
test_samples = []

train_groups = set()
val_groups = set()
test_groups = set()

for class_name in classes:
    class_groups = list(
        groups_by_class[class_name].items()
    )

    rng.shuffle(class_groups)

    number_of_groups = len(class_groups)

    number_of_test_groups = max(
        1,
        round(number_of_groups * 0.10),
    )

    number_of_val_groups = max(
        1,
        round(number_of_groups * 0.10),
    )

    while (
        number_of_test_groups
        + number_of_val_groups
        >= number_of_groups
    ):
        if number_of_test_groups > 1:
            number_of_test_groups -= 1
        elif number_of_val_groups > 1:
            number_of_val_groups -= 1
        else:
            break

    test_part = class_groups[
        :number_of_test_groups
    ]

    val_part = class_groups[
        number_of_test_groups:
        number_of_test_groups
        + number_of_val_groups
    ]

    train_part = class_groups[
        number_of_test_groups
        + number_of_val_groups:
    ]

    for group_id, samples in train_part:
        train_groups.add(group_id)
        train_samples.extend(samples)

    for group_id, samples in val_part:
        val_groups.add(group_id)
        val_samples.extend(samples)

    for group_id, samples in test_part:
        test_groups.add(group_id)
        test_samples.extend(samples)


rng.shuffle(train_samples)
rng.shuffle(val_samples)
rng.shuffle(test_samples)

assert len(train_samples) == 2578
assert len(val_samples) == 335
assert len(test_samples) == 335

assert train_groups.isdisjoint(val_groups)
assert train_groups.isdisjoint(test_groups)
assert val_groups.isdisjoint(test_groups)

# Ajouter les 55 vidéos uniquement au train
new_label = class_to_idx["objects_falling"]

added_train_samples = [
    (path, new_label)
    for path in added_videos
]

train_samples.extend(added_train_samples)

# Nouveau mélange reproductible
rng.shuffle(train_samples)

assert len(train_samples) == 2633
assert len(val_samples) == 335
assert len(test_samples) == 335

# ============================================================
# CRÉER DES ALIAS UNIQUES POUR LES VIDÉOS
# ============================================================

all_samples = (
    train_samples
    + val_samples
    + test_samples
)

sample_to_alias = {}

for index, (path, label) in enumerate(all_samples):
    class_name = path.relative_to(NEW_ROOT).parts[0]
    extension = path.suffix.lower()

    alias = (
        f"rhad_training2_{index:05d}_"
        f"{class_name}{extension}"
    )

    alias_path = VIDEO_ROOT / alias

    os.symlink(
        str(path),
        str(alias_path),
    )

    sample_to_alias[str(path)] = alias

assert len(sample_to_alias) == 3303

# ============================================================
# ÉCRIRE LES ANNOTATIONS
# ============================================================

def write_annotations(destination, samples):
    with destination.open(
        "w",
        encoding="utf-8",
    ) as file:
        for path, label in samples:
            file.write(
                f"{sample_to_alias[str(path)]} {label}\n"
            )


write_annotations(
    ANNOTATIONS_DIR / "train.csv",
    train_samples,
)

write_annotations(
    ANNOTATIONS_DIR / "val.csv",
    val_samples,
)

write_annotations(
    ANNOTATIONS_DIR / "test.csv",
    test_samples,
)

with (
    ANNOTATIONS_DIR / "label_mapping.csv"
).open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.writer(file)
    writer.writerow(["label", "class_name"])

    for class_name, label in class_to_idx.items():
        writer.writerow([label, class_name])


summary = {
    "old_train": 2578,
    "new_videos_added_to_train": 55,
    "new_train": len(train_samples),
    "validation": len(val_samples),
    "test": len(test_samples),
    "class_to_idx": class_to_idx,
}

(
    ANNOTATIONS_DIR / "training2_split_summary.json"
).write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8",
)

print("\n" + "=" * 70)
print("SPLIT FINAL POUR LE FINE-TUNING")
print("=" * 70)
print("Train avant ajout :", 2578)
print("Nouvelles vidéos  :", 55)
print("Train final        :", len(train_samples))
print("Validation         :", len(val_samples))
print("Test               :", len(test_samples))
print("Fuites             : 0")
print("Label objects_falling :", new_label)

print("\nANNOTATIONS CRÉÉES")


In [ ]:
# ============================================================
# CONTRÔLE DES 55 NOUVELLES VIDÉOS
# ============================================================

numbers = []

for path in added_videos:
    match = re.fullmatch(
        r"1 \((\d+)\)\.mp4",
        path.name,
    )

    assert match is not None
    numbers.append(int(match.group(1)))

expected_numbers = list(range(61, 116))

assert numbers == expected_numbers, (
    "La suite des fichiers n'est pas complète "
    "entre 61 et 115."
)

print("Première vidéo :", added_videos[0].name)
print("Dernière vidéo :", added_videos[-1].name)
print("Total :", len(added_videos))

print(
    "Les 55 nouvelles vidéos sont correctement détectées"
)

In [ ]:
# ============================================================
# SMOKE TEST DU NOUVEAU DATASET
# ============================================================

from pathlib import Path
import os
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"
REPO_DIR = Path("/kaggle/working/VideoMAEv2")

test_code = r"""
from pathlib import Path
from types import SimpleNamespace

import torch
import models
from dataset import build_dataset


ANNOTATIONS_DIR = Path(
    "/kaggle/working/rhad_videomae_training2_annotations"
)

VIDEO_ROOT = Path(
    "/kaggle/working/rhad_videomae_training2_videos"
)


args = SimpleNamespace(
    # Dataset
    data_set="RHAD",
    data_path=str(ANNOTATIONS_DIR),
    data_root=str(VIDEO_ROOT),
    nb_classes=12,

    # Vidéo
    input_size=224,
    short_side_size=224,
    num_frames=16,
    num_segments=1,
    tubelet_size=2,
    sampling_rate=4,

    # Test
    test_num_segment=2,
    test_num_crop=3,

    # Augmentations d'entraînement
    aa="rand-m7-n4-mstd0.5-inc1",
    train_interpolation="bicubic",
    repeated_aug=False,
    reprob=0.25,
    remode="pixel",
    recount=1,
    resplit=False,

    # Mixup / CutMix
    mixup=0.8,
    cutmix=1.0,
    cutmix_minmax=None,
    mixup_prob=1.0,
    mixup_switch_prob=0.5,
    mixup_mode="batch",
    smoothing=0.1,

    # Autres paramètres parfois attendus
    color_jitter=0.4,
    num_sample=1,
    imagenet_default_mean_and_std=True,
)


train_dataset, number_of_classes = build_dataset(
    is_train=True,
    test_mode=False,
    args=args,
)

print("Taille train :", len(train_dataset))
print("Nombre de classes :", number_of_classes)

sample = train_dataset[0]

if not isinstance(sample, tuple):
    raise TypeError(
        f"Type inattendu pour un échantillon : {type(sample)}"
    )

video = sample[0]
label = sample[1]

print("Type vidéo  :", type(video))
print("Shape vidéo :", tuple(video.shape))
print("Label       :", label)

assert len(train_dataset) == 2633, (
    f"2633 vidéos attendues, mais {len(train_dataset)} trouvées."
)

assert torch.isfinite(video).all(), (
    "La vidéo contient des valeurs NaN ou infinies."
)

print("SMOKE TEST RÉUSSI")
"""


environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0"

subprocess.run(
    [PY, "-c", test_code],
    cwd=str(REPO_DIR),
    env=environment,
    check=True,
)

In [ ]:
# ============================================================
# FINE-TUNING VIDEOMAE SUR TRAINING2
# ============================================================

from pathlib import Path
import datetime
import os
import shutil
import subprocess

PY = "/kaggle/working/vmae_env/bin/python"
REPO_DIR = Path("/kaggle/working/VideoMAEv2")

ANNOTATIONS_DIR = Path(
    "/kaggle/working/rhad_videomae_training2_annotations"
)

VIDEO_ROOT = Path(
    "/kaggle/working/rhad_videomae_training2_videos"
)

OUTPUT_DIR = Path(
    "/kaggle/working/videomae_training2_finetuning"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for path in [
    Path(PY),
    REPO_DIR,
    ANNOTATIONS_DIR / "train.csv",
    ANNOTATIONS_DIR / "val.csv",
    ANNOTATIONS_DIR / "test.csv",
    VIDEO_ROOT,
    OLD_BEST_CHECKPOINT,
]:
    assert path.exists(), path

process_check = subprocess.run(
    [
        "bash",
        "-lc",
        "ps aux | grep run_class_finetuning.py "
        "| grep -v grep || true",
    ],
    text=True,
    capture_output=True,
)

if process_check.stdout.strip():
    print(process_check.stdout)
    raise RuntimeError(
        "Un entraînement VideoMAE est déjà actif."
    )

command = [
    PY,
    "-m",
    "torch.distributed.run",
    "--nproc_per_node=2",
    "--master_port=29531",
    "run_class_finetuning.py",

    # Architecture identique
    "--model", "vit_base_patch16_224",
    "--nb_classes", "12",
    "--use_mean_pooling",
    "--init_scale", "0.001",

    # Dataset Training2
    "--data_set", "RHAD",
    "--data_path", str(ANNOTATIONS_DIR),
    "--data_root", str(VIDEO_ROOT),

    # Configuration vidéo identique
    "--input_size", "224",
    "--short_side_size", "224",
    "--num_frames", "16",
    "--num_segments", "1",
    "--tubelet_size", "2",
    "--sampling_rate", "4",

    # Fine-tuning court
    "--epochs", "5",
    "--start_epoch", "0",

    "--batch_size", "2",
    "--update_freq", "1",
    "--num_workers", "2",
    "--seed", "42",

    # Learning rate faible pour conserver l'ancien apprentissage
    "--opt", "adamw",
    "--lr", "1e-5",
    "--min_lr", "1e-7",
    "--weight_decay", "0.05",
    "--layer_decay", "0.75",
    "--warmup_epochs", "1",
    "--clip_grad", "5.0",
    "--opt_betas", "0.9", "0.999",
    "--drop_path", "0.1",

    # Augmentations identiques à l'ancien entraînement
    "--num_sample", "1",
    "--smoothing", "0.1",
    "--mixup", "0.8",
    "--cutmix", "1.0",
    "--mixup_prob", "1.0",
    "--mixup_switch_prob", "0.5",
    "--color_jitter", "0.4",
    "--aa", "rand-m7-n4-mstd0.5-inc1",
    "--reprob", "0.25",
    "--remode", "pixel",
    "--recount", "1",

    "--test_num_segment", "2",
    "--test_num_crop", "3",

    # Charger les poids de l'ancien meilleur modèle
    "--finetune", str(OLD_BEST_CHECKPOINT),
    "--model_key", "model",
    "--no_auto_resume",

    "--output_dir", str(OUTPUT_DIR),
    "--log_dir", str(OUTPUT_DIR),
    "--save_ckpt",
    "--save_ckpt_freq", "1",
]

assert "--finetune" in command
assert "--resume" not in command

environment = os.environ.copy()
environment["CUDA_VISIBLE_DEVICES"] = "0,1"
environment["OMP_NUM_THREADS"] = "1"
environment["PYTHONUNBUFFERED"] = "1"
environment["TOKENIZERS_PARALLELISM"] = "false"

timestamp = datetime.datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

training_log = OUTPUT_DIR / (
    f"training2_finetuning_{timestamp}.log"
)

print("=" * 70)
print("FINE-TUNING VIDEOMAE")
print("=" * 70)
print("Ancien checkpoint :", OLD_BEST_CHECKPOINT)
print("Train             : 2633 vidéos")
print("Validation        : 335 vidéos")
print("Test              : 335 vidéos")
print("Epochs            : 5")
print("Learning rate     : 1e-5")
print("Sortie            :", OUTPUT_DIR)
print("=" * 70 + "\n")

with training_log.open(
    "w",
    encoding="utf-8",
) as log_file:

    process = subprocess.Popen(
        command,
        cwd=str(REPO_DIR),
        env=environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="", flush=True)
        log_file.write(line)
        log_file.flush()

    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Le fine-tuning s'est arrêté avec le code {return_code}."
    )

print("\nFINE-TUNING TERMINÉ")


In [ ]:
# ============================================================
# VÉRIFIER LE NOUVEAU MEILLEUR CHECKPOINT
# ============================================================

from pathlib import Path
import json
import torch

OUTPUT_DIR = Path(
    "/kaggle/working/videomae_training2_finetuning"
)

BEST = OUTPUT_DIR / "checkpoint-best.pth"
LATEST = OUTPUT_DIR / "checkpoint-latest.pth"
LOG_FILE = OUTPUT_DIR / "log.txt"

for path in [BEST, LATEST, LOG_FILE]:
    print(
        path.name,
        "->",
        path.exists(),
        (
            f"({path.stat().st_size / 1024**2:.2f} MiB)"
            if path.exists()
            else ""
        ),
    )

assert BEST.exists()
assert LATEST.exists()

best_checkpoint = torch.load(
    BEST,
    map_location="cpu",
    weights_only=False,
)

assert tuple(
    best_checkpoint["model"]["head.weight"].shape
) == (12, 768)

print("\nEpoch du nouveau checkpoint-best :", best_checkpoint.get("epoch"))

records = []

if LOG_FILE.exists():
    for line in LOG_FILE.read_text(
        encoding="utf-8",
        errors="ignore",
    ).splitlines():
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            pass

rows = []

for record in records:
    accuracy = record.get(
        "test_acc1",
        record.get("val_acc1"),
    )

    if accuracy is not None:
        rows.append(
            {
                "epoch": record.get("epoch"),
                "accuracy": float(accuracy),
            }
        )

if rows:
    best_row = max(
        rows,
        key=lambda row: row["accuracy"],
    )

    print("Meilleure epoch selon le log :", best_row["epoch"])
    print("Meilleure validation Acc@1  :", best_row["accuracy"])

print("\nNOUVEAU MODÈLE PRÊT")


In [ ]:
# ============================================================
# CRÉER LE DATASET KAGGLE DU NOUVEAU MODÈLE VIDEOMAE
# ============================================================

from pathlib import Path
import json
import shutil
import subprocess


# ------------------------------------------------------------
# 1. DOSSIER PRODUIT PAR LE FINE-TUNING
# ------------------------------------------------------------

SOURCE_DIR = Path(
    "/kaggle/working/videomae_training2_finetuning"
)

assert SOURCE_DIR.exists(), (
    f"Dossier d'entraînement introuvable : {SOURCE_DIR}"
)


# ------------------------------------------------------------
# 2. FICHIERS À CONSERVER
# ------------------------------------------------------------

FILES_TO_SAVE = [
    "checkpoint-best.pth",
    "checkpoint-latest.pth",
    "log.txt",
]

for filename in FILES_TO_SAVE:
    source_file = SOURCE_DIR / filename

    assert source_file.exists(), (
        f"Fichier introuvable : {source_file}"
    )

    print(
        filename,
        "->",
        source_file.exists(),
        f"({source_file.stat().st_size / 1024**2:.2f} MiB)",
    )


# ------------------------------------------------------------
# 3. CRÉATION DU DOSSIER FINAL DU DATASET
# ------------------------------------------------------------

DATASET_DIR = Path(
    "/kaggle/working/"
    "videomae-training2-finetuned-results"
)

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

for filename in FILES_TO_SAVE:
    shutil.copy2(
        SOURCE_DIR / filename,
        DATASET_DIR / filename,
    )


# ------------------------------------------------------------
# 4. AJOUT D'UN FICHIER D'INFORMATIONS
# ------------------------------------------------------------

information = {
    "model": "VideoMAE",
    "task": "Child hazardous activity recognition",
    "number_of_classes": 12,
    "training_dataset": "Training2",
    "old_videos": 3248,
    "added_videos": 55,
    "total_videos": 3303,
    "train_videos": 2633,
    "validation_videos": 335,
    "test_videos": 335,
    "fine_tuning_epochs": 5,
    "learning_rate": 1e-5,
    "best_validation_acc1": 93.13432933750437,
    "best_epoch": 0,
    "main_checkpoint": "checkpoint-best.pth",
}

with (
    DATASET_DIR / "model_information.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        information,
        file,
        indent=4,
        ensure_ascii=False,
    )


# ------------------------------------------------------------
# 5. MÉTADONNÉES KAGGLE
# ------------------------------------------------------------

metadata = {
    "title": "VideoMAE Training2 Finetuned Results",
    "id": (
        "nisrineamalkhnifass/"
        "videomae-training2-finetuned-results"
    ),
    "licenses": [
        {
            "name": "CC0-1.0"
        }
    ],
}

with (
    DATASET_DIR / "dataset-metadata.json"
).open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        indent=4,
    )


# ------------------------------------------------------------
# 6. AFFICHAGE DU CONTENU
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("DOSSIER DU DATASET PRÊT")
print("=" * 70)

total_size = 0

for file_path in sorted(DATASET_DIR.iterdir()):
    size_mib = file_path.stat().st_size / 1024**2
    total_size += file_path.stat().st_size

    print(
        f"{file_path.name:30s}",
        f"{size_mib:.2f} MiB",
    )

print(
    "\nTaille totale :",
    f"{total_size / 1024**3:.2f} GiB",
)


# ------------------------------------------------------------
# 7. TENTATIVE DE CRÉATION DIRECTE SUR KAGGLE
# ------------------------------------------------------------

command = [
    "kaggle",
    "datasets",
    "create",
    "-p",
    str(DATASET_DIR),
    "--public",
]

print("\nCommande exécutée :")
print(" ".join(command))

result = subprocess.run(
    command,
    text=True,
    capture_output=True,
)

print("\nSortie Kaggle :")
print(result.stdout)

if result.returncode == 0:
    print(
        "\nDATASET KAGGLE CRÉÉ AVEC SUCCÈS"
    )
    print(
        "Nom : "
        "nisrineamalkhnifass/"
        "videomae-training2-finetuned-results"
    )
else:
    print("\nATTENTION : Création automatique non effectuée.")
    print(result.stderr)

    print(
        "\nLe dossier est quand même prêt ici :"
    )
    print(DATASET_DIR)

    print(
        "\nDans Kaggle : Save Version → "
        "Save & Run All → Create Dataset from Output."
    )

## Après l’entraînement

Le fichier principal à conserver est :

```text
/kaggle/working/videomae_training2_finetuning/checkpoint-best.pth
```

Ne remplace pas encore définitivement l’ancien modèle avant d’avoir comparé les deux modèles sur le même jeu de test de 335 vidéos.
